# 🏦 AML Transaction Monitoring Model
**Author:** Anushka Shinde | MS Finance, Boston University  
**Stage 2:** Rule-Based AML Red Flags

---
In Stage 1, we built a dataset of 500 transactions.  
In Stage 2, we write **AML rules** that flag suspicious transactions.

These rules are based on real **AML typologies** — patterns that compliance teams at banks actually look for.  
You learned about many of these at Deloitte!

> **What is a typology?**  
> A typology is a known pattern or method used by money launderers. Regulators like FATF publish typology reports so banks know what to look for.

## Step 1: Re-run Stage 1 Code
We need to rebuild our dataset first. Copy and run this — it is the same code from Stage 1.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

customer_types = ['Individual', 'Business', 'Shell Company', 'NGO']
countries = ['USA', 'UK', 'India', 'Cayman Islands', 'Panama',
             'Switzerland', 'Germany', 'UAE', 'Nigeria', 'Singapore']
high_risk_countries = ['Cayman Islands', 'Panama', 'Nigeria']
transaction_types = ['Wire Transfer', 'Cash Deposit', 'ATM Withdrawal',
                     'Online Transfer', 'Check', 'Crypto Exchange']

n = 500

df = pd.DataFrame({
    'Transaction_ID': [f'TXN{str(i).zfill(5)}' for i in range(1, n + 1)],
    'Customer_ID': [f'CUST{np.random.randint(1000, 2000)}' for _ in range(n)],
    'Customer_Type': np.random.choice(customer_types, n, p=[0.5, 0.3, 0.1, 0.1]),
    'Transaction_Amount': np.round(
        np.where(
            np.random.rand(n) > 0.95,
            np.random.uniform(9000, 9999, n),
            np.random.exponential(scale=3000, size=n).clip(100, 100000)
        ), 2),
    'Transaction_Type': np.random.choice(transaction_types, n),
    'Origin_Country': np.random.choice(countries, n,
        p=[0.3, 0.15, 0.15, 0.05, 0.05, 0.08, 0.1, 0.05, 0.04, 0.03]),
    'Destination_Country': np.random.choice(countries, n),
    'Num_Transactions_Last_30Days': np.random.randint(1, 50, n),
    'Avg_Transaction_Last_6Months': np.round(
        np.random.exponential(scale=2000, size=n).clip(100, 50000), 2),
    'Account_Age_Years': np.round(np.random.uniform(0.1, 20, n), 1),
    'Prior_SAR_Filed': np.random.choice([0, 1], n, p=[0.92, 0.08]),
})

print(f'✅ Dataset ready — {len(df)} transactions loaded.')

---
## Step 2: Understand What We Are Building

We are going to write a **function** that looks at each transaction row and asks:  
*"Does this transaction match any known suspicious pattern?"*

If yes → it adds a label like `"Structuring"` or `"High-Risk Country"` to that row.

We will check for **7 red flags**, each based on a real AML typology:

| # | Red Flag | What it Detects |
|---|---|---|
| 1 | Structuring | Amounts just below $10,000 to avoid CTR |
| 2 | High-Risk Country | Transactions involving FATF-listed jurisdictions |
| 3 | Unusual Amount vs History | Amount far higher than customer's normal behavior |
| 4 | High-Risk Entity Type | Shell companies or NGOs doing wire transfers |
| 5 | Prior SAR on File | Customer was previously reported as suspicious |
| 6 | New Account High Value | Brand new account making large transactions |
| 7 | Excessive Frequency | Too many transactions in a short period (smurfing) |

---
## Step 3: Write the Red Flag Function

A **function** in Python is a reusable block of code. We define it once with `def`, and then apply it to every row in our dataset.

Think of it like an Excel formula — you write it once and drag it down all 500 rows.

Read each rule carefully — the comments explain the real-world AML reasoning behind each one.

In [ ]:
def apply_red_flags(row):
    """
    This function takes ONE transaction row as input.
    It checks it against 7 AML rules.
    It returns a string listing all the red flags found.
    If no flags, it returns 'None'.
    """

    # Start with an empty list — we'll add flags as we find them
    flags = []

    # --------------------------------------------------------
    # RULE 1: STRUCTURING
    # --------------------------------------------------------
    # Amounts between $9,000 and $9,999 are suspicious.
    # This is called structuring — deliberately keeping amounts
    # just below $10,000 to avoid triggering a CTR filing.
    # It is a federal crime in the USA (31 U.S.C. § 5324).
    if 9000 <= row['Transaction_Amount'] <= 9999:
        flags.append('Structuring')

    # --------------------------------------------------------
    # RULE 2: HIGH-RISK COUNTRY
    # --------------------------------------------------------
    # If the money is coming FROM or going TO a high-risk country,
    # it needs extra scrutiny. These jurisdictions have weak AML
    # controls and are commonly used for layering.
    if row['Origin_Country'] in high_risk_countries or \
       row['Destination_Country'] in high_risk_countries:
        flags.append('High-Risk Country')

    # --------------------------------------------------------
    # RULE 3: UNUSUAL AMOUNT VS HISTORY
    # --------------------------------------------------------
    # If this transaction is MORE THAN 5x the customer's
    # average transaction, something unusual is happening.
    # Example: Customer usually sends $500, but today sends $8,000.
    # This is a classic placement or layering indicator.
    if row['Avg_Transaction_Last_6Months'] > 0:
        ratio = row['Transaction_Amount'] / row['Avg_Transaction_Last_6Months']
        if ratio > 5:
            flags.append('Unusual Amount vs History')

    # --------------------------------------------------------
    # RULE 4: HIGH-RISK ENTITY TYPE
    # --------------------------------------------------------
    # Shell companies and NGOs are commonly used to move
    # illicit funds. When they use wire transfers (which are
    # harder to trace), that combination is a strong red flag.
    if row['Customer_Type'] in ['Shell Company', 'NGO'] and \
       row['Transaction_Type'] == 'Wire Transfer':
        flags.append('High-Risk Entity Type')

    # --------------------------------------------------------
    # RULE 5: PRIOR SAR ON FILE
    # --------------------------------------------------------
    # If a SAR was already filed for this customer, any new
    # transaction from them carries elevated inherent risk.
    if row['Prior_SAR_Filed'] == 1:
        flags.append('Prior SAR on File')

    # --------------------------------------------------------
    # RULE 6: NEW ACCOUNT HIGH VALUE
    # --------------------------------------------------------
    # An account less than 1 year old making transactions
    # above $10,000 is suspicious. New accounts are often
    # opened specifically to move illicit funds quickly.
    if row['Account_Age_Years'] < 1 and row['Transaction_Amount'] > 10000:
        flags.append('New Account High Value')

    # --------------------------------------------------------
    # RULE 7: EXCESSIVE TRANSACTION FREQUENCY
    # --------------------------------------------------------
    # More than 30 transactions in 30 days is unusual for
    # most customers. This pattern is linked to SMURFING —
    # breaking one large amount into many small transactions
    # across multiple accounts to avoid detection.
    if row['Num_Transactions_Last_30Days'] > 30:
        flags.append('Excessive Transaction Frequency')

    # --------------------------------------------------------
    # RETURN THE RESULT
    # --------------------------------------------------------
    # If we found flags, join them with a semicolon separator.
    # If no flags were found, return 'None'.
    if len(flags) > 0:
        return '; '.join(flags)
    else:
        return 'None'

print('✅ Red flag function defined — ready to apply!')

---
## Step 4: Apply the Function to All 500 Transactions

`df.apply()` runs our function on **every single row** in the dataset automatically.  
`axis=1` means "apply row by row" (axis=0 would be column by column).

This is the equivalent of dragging a formula down 500 rows in Excel.

In [ ]:
# Apply the red flag function to every row
df['Red_Flags'] = df.apply(apply_red_flags, axis=1)

# Count how many flags each transaction has
# This will be useful later for the ML model
df['Flag_Count'] = df['Red_Flags'].apply(
    lambda x: 0 if x == 'None' else len(x.split(';'))
)

print('✅ Red flags applied to all transactions!')
print(f'\n   Transactions with at least 1 flag : {(df["Flag_Count"] > 0).sum()}')
print(f'   Transactions with no flags        : {(df["Flag_Count"] == 0).sum()}')

---
## Step 5: Analyse the Results

In [ ]:
# Preview flagged transactions
print('🚨 Sample of flagged transactions:')
df[df['Flag_Count'] > 0][['Transaction_ID', 'Customer_Type', 'Transaction_Amount',
                            'Origin_Country', 'Red_Flags', 'Flag_Count']].head(10)

In [ ]:
# How many times does each red flag appear?
print('📊 Red Flag Frequency:')

flag_types = [
    'Structuring',
    'High-Risk Country',
    'Unusual Amount vs History',
    'High-Risk Entity Type',
    'Prior SAR on File',
    'New Account High Value',
    'Excessive Transaction Frequency'
]

for flag in flag_types:
    count = df['Red_Flags'].str.contains(flag).sum()
    pct = round(count / len(df) * 100, 1)
    print(f'   {flag:<35} : {count:>4} transactions ({pct}%)')

In [ ]:
# How many flags does a single transaction have at most?
print('🔢 Flag Count Distribution:')
print(df['Flag_Count'].value_counts().sort_index())

print(f'\n   Max flags on one transaction : {df["Flag_Count"].max()}')
print(f'   Avg flags (flagged only)     : {df[df["Flag_Count"]>0]["Flag_Count"].mean():.2f}')

In [ ]:
# Show the most suspicious transactions (most flags)
print('🔴 Top 10 Most Flagged Transactions:')
df.sort_values('Flag_Count', ascending=False)[
    ['Transaction_ID', 'Customer_ID', 'Customer_Type',
     'Transaction_Amount', 'Origin_Country', 'Red_Flags', 'Flag_Count']
].head(10)

---
## ✅ Stage 2 Complete!

You have now added **7 AML red flags** to every transaction in your dataset.

### What you just built mirrors real compliance work:
- Banks run rules like these automatically on every transaction, 24/7
- When a rule fires, it creates an **alert** in the bank's transaction monitoring system (e.g. Actimize, Mantas, NICE)
- A compliance analyst (like the role you're applying for!) then reviews the alert and decides whether to file a SAR

---
### Before moving to Stage 3, answer these:

1. **What is structuring and why is it illegal?**
2. **What is smurfing?** (Hint: Rule 7)
3. **Why are shell companies + wire transfers a red flag combination?**
4. **Look at your output — which red flag appeared the most? Why do you think that is?**

---
**Next → Stage 3: ML Risk Scoring**  
We will train an Isolation Forest model to give every transaction a **risk score from 0–100** based on how anomalous it looks — going beyond simple rules.